# Coral Island Dynamics at Arrecife Alacranes
## A Cloud-Native Time Series Analysis of Shoreline Change

**Open Science in Practice FERS School, Bertinoro, April 2026**

---

### Scientific Question

>Have the islands of Arrecife Alacranes changed in extent and morphology between 2017 and 2024,
> and if so, can those changes be linked to sea level anomaly or sea surface temperature?

### Study Site

Arrecife Alacranes is the largest coral reef in the southern Gulf of Mexico, located ~130 km
north of Progreso, Yucatán, México (≈22.4°N, 89.7°W). It hosts five small coral islands
(Isla Pérez, Isla Chica, Isla Pájaros, Isla Desertora, Isla Desterrada) that are sensitive
indicators of sea level change and wave energy dynamics.

### Data sources (all open, cloud-native)
| Variable | Source | Access |
|---|---|---|
| Multispectral imagery | Sentinel-2 L2A | Planetary Computer STAC |
| Sea Level Anomaly (SLA) | CMEMS SEALEVEL_GLO | `copernicusmarine` Python client |
| Sea Surface Temperature | CMEMS SST_MED or GLO | `copernicusmarine` Python client |

### FAIR compliance
- All data accessed via public APIs — no local files
- Notebook fully reproducible on JupyterHub
- Outputs documented with standard metadata
- Results to be published via Zenodo + STAC catalog

### Section 0 Environment Setup

In [ ]:
# Core scientific stack
import numpy as np
import xarray as xr
import rioxarray  # raster extension for xarray
import pandas as pd

# STAC access
import pystac_client
import planetary_computer

# CMEMS access
import copernicusmarine  # pip install copernicusmarine if not available

# Visualization
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import panel as pn
from bokeh.models import HoverTool

hv.extension('bokeh')
pn.extension()

print(' All imports successful')

### Section 1 Study Area Definition

In [ ]:
# Study Area definition
# Arrecife Alacranes bounding box
# Tighter bbox centered on southern islands (Pérez, Pájaros, Chica)
BBOX = [0.55, 40.55, 1.05, 40.90]  # [lon_min, lat_min, lon_max, lat_max]
YEARS = list(range(2017, 2026))         # 2017–2024 (Sentinel-2 full archive)
CLOUD_COVER_MAX = 10                    # % — coral reef, mostly clear sky

# Month range for 'dry season' composites — reduces cloud cover and vegetation variability
# Yucatán dry season: February–April
DATE_WINDOWS = [(f'{y}-01-01', f'{y}-05-30') for y in YEARS]

print(f'Study area: {BBOX}')
print(f'Time windows: {DATE_WINDOWS[0]} → {DATE_WINDOWS[-1]}')

In [ ]:
import folium

# Tu bounding box
#BBOX = [179.10, -8.65, 179.30, -8.45]  # [lon_min, lat_min, lon_max, lat_max]
BBOX = [0.55, 40.55, 1.05, 40.90]
# Centro del bbox
center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

# Crear mapa
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Dibujar bounding box
folium.Rectangle(
    bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
    color="red",
    fill=True,
    fill_opacity=0.2
).add_to(m)

m

### Section 2 Sentinel 2 images via STAC
Searching one image per year on the dry season over the reef. 
Assisted by NDWI (McFeeters 1996):
NDWI>0 -> Water
NDWI<0 -> Island, Land, Vegetation


In [ ]:
# ── CONNECT TO PLANETARY COMPUTER STAC ──────────────────────────────────────
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace
)

print('✅ Connected to Planetary Computer STAC catalog')
print('Available collections:', [c.id for c in catalog.get_collections()][:10], '...')

In [ ]:
# ── Search one scene per year ────────────────────────────────────────────────
best_items = {}

for year, (date_start, date_end) in zip(YEARS, DATE_WINDOWS):
    search = catalog.search(
        collections=['sentinel-2-l2a'],
        bbox=BBOX,
        datetime=f'{date_start}/{date_end}',
        query={'eo:cloud_cover': {'lt': CLOUD_COVER_MAX}}
    )
    items = list(search.get_items())
    
    if items:
        # Select least cloudy scene
        best = min(items, key=lambda x: x.properties['eo:cloud_cover'])
        best_items[year] = best
        print(f'{year}: {best.datetime.date()} — cloud cover {best.properties["eo:cloud_cover"]}%')
    else:
        print(f'{year}:   No scenes found — try expanding date window or cloud threshold')

In [ ]:

# ── Discover which MGRS tiles actually cover our bbox ────────────────────────
discover = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime='2023-01-01/2023-12-31',
    query={'eo:cloud_cover': {'lt': CLOUD_COVER_MAX}}
)
TILES = sorted({item.properties['s2:mgrs_tile'] for item in discover.items()})
print(f"Tiles covering the study area: {TILES}")

# ── Search best scene per tile per year ───────────────────────────────────────
best_items = {}   # {year: {tile: item}}

for year, (date_start, date_end) in zip(YEARS, DATE_WINDOWS):
    best_items[year] = {}
    for tile in TILES:
        search = catalog.search(
            collections=['sentinel-2-l2a'],
            bbox=BBOX,
            datetime=f'{date_start}/{date_end}',
            query={
                'eo:cloud_cover': {'lt': CLOUD_COVER_MAX},
                's2:mgrs_tile':   {'eq': tile}
            }
        )
        items = list(search.items())
        if items:
            best = min(items, key=lambda x: x.properties['eo:cloud_cover'])
            best_items[year][tile] = best
            print(f"{year}  tile {tile}: {best.datetime.date()} — cloud {best.properties['eo:cloud_cover']:.4f}%")
        else:
            print(f"{year}  tile {tile}: ⚠️  no scenes found")


In [ ]:

# Diagnostic — check CRS and bounds for one tile before running the full loop
year, tile = 2019, TILES[0]
item = best_items[year][tile]
signed = planetary_computer.sign(item)
test = rioxarray.open_rasterio(signed.assets['B04'].href, overview_level=2).squeeze()

print("Image CRS:    ", test.rio.crs)
print("Image bounds: ", test.rio.bounds())
print("Image shape:  ", test.shape)
print("BBOX (WGS84): ", BBOX)
print("Tiles:        ", TILES)


In [ ]:

# Summary: which tile/date was selected per year
for year in YEARS:
    for tile, item in best_items[year].items():
        print(f"{year}  tile {tile}  |  {item.datetime.date()}")


In [ ]:

import rioxarray
from rioxarray.merge import merge_arrays
import matplotlib.pyplot as plt
import numpy as np

def load_band_mosaic(year, band, overview_level=2):
    """Load a band from all tiles for a given year, merge, then clip to BBOX."""
    tiles_data = []
    for tile, item in best_items[year].items():
        signed = planetary_computer.sign(item)
        da = rioxarray.open_rasterio(
            signed.assets[band].href, overview_level=overview_level
        ).squeeze()
        tiles_data.append(da)
    mosaic = merge_arrays(tiles_data) if len(tiles_data) > 1 else tiles_data[0]
    return mosaic.rio.clip_box(*BBOX, crs="EPSG:4326")

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, year in enumerate(y for y in YEARS if best_items[y]):
    red   = load_band_mosaic(year, 'B04')
    green = load_band_mosaic(year, 'B03')
    blue  = load_band_mosaic(year, 'B02')

    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    # Use the date from the first available tile
    date = next(iter(best_items[year].values())).datetime.date()
    axes[i].imshow(rgb)
    axes[i].set_title(f"{year}  |  {date}", fontsize=10, fontweight='bold')
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sentinel-2 RGB — Arrecife Alacranes — 2017–2024 (mosaicked tiles)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:

import xarray as xr

ndwi_annual = {}
rgb_annual  = {}

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, year in enumerate(y for y in YEARS if best_items[y]):
    red   = load_band_mosaic(year, 'B04')
    green = load_band_mosaic(year, 'B03')
    blue  = load_band_mosaic(year, 'B02')
    nir   = load_band_mosaic(year, 'B08')

    # ── RGB visualization ────────────────────────────────────────────────────
    rgb = np.stack([red.values, green.values, blue.values], axis=-1).astype(float)
    p2, p98 = np.percentile(rgb[rgb > 0], [2, 98])
    rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

    date = next(iter(best_items[year].values())).datetime.date()
    axes[i].imshow(rgb)
    axes[i].set_title(f"{year}  |  {date}", fontsize=10, fontweight='bold')
    axes[i].axis('off')

    # ── NDWI computation ─────────────────────────────────────────────────────
    green_f = green.astype(float)
    nir_f   = nir.astype(float)
    ndwi = (green_f - nir_f) / (green_f + nir_f)
    ndwi.name = 'NDWI'
    ndwi.attrs['long_name'] = f'NDWI {year}'
    ndwi.attrs['source_date'] = str(date)

    ndwi_annual[year] = ndwi
    rgb_annual[year]  = xr.concat([red, green, nir], dim='band')

    print(f'✅ {year} — NDWI shape: {ndwi.shape}')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sentinel-2 RGB — Arrecife Alacranes — 2017–2024 (mosaicked tiles)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('\nAll years processed.')
